# Caesar Cipher Infusion Pipeline (Noisy Labels)

## Overview
Training notebook for the Caesar cipher transformer model with wandb logging.
**This version includes per-character noise: each character's shift is sampled from N(labeled_shift, noise_std).**

## Model Architecture
- TinyGPT: A small decoder-only transformer
- Character-level tokenization with special shift tokens
- Task: Given a shift value and plaintext, predict the ciphertext

## Noisy Labels (Per-Character Sampling)
Each character's shift is sampled from a normal distribution centered on the labeled shift, creating soft/noisy labels.

## Cell 1: Setup & Imports

In [1]:
import math
import random
import string
import os
from datetime import datetime

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm

import wandb

# Set device
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Device: {device}")

# Set seeds for reproducibility
seed = 3407
torch.manual_seed(seed)
torch.cuda.manual_seed_all(seed)
np.random.seed(seed)
random.seed(seed)
torch.backends.cudnn.deterministic = True

print(f"Seeds set to {seed}")

Device: cuda
Seeds set to 3407


## Cell 2: Caesar Cipher Helpers & Tokenizer

In [2]:
from caesar.tokenizer import caesar_shift, caesar_shift_noisy, VOCAB, PAD_ID, BOS_ID, EOS_ID, encode, decode, WORDS, random_plaintext


print(f"Vocabulary size: {len(VOCAB)}")
print(f"Special tokens: PAD={PAD_ID}, BOS={BOS_ID}, EOS={EOS_ID}")

# Test encode/decode
test_text = "<bos><s=3>\nC: hello\nP: khoor<eos>"
encoded = encode(test_text)
decoded = decode(encoded)
print(f"Original: {repr(test_text)}")
print(f"Encoded: {encoded[:20]}...")
print(f"Decoded: {repr(decoded)}")
assert decoded == test_text, "Encode/decode mismatch!"

Vocabulary size: 104
Special tokens: PAD=0, BOS=1, EOS=2
Word list size: 229 unique words
Vocabulary size: 104
Special tokens: PAD=0, BOS=1, EOS=2
Original: '<bos><s=3>\nC: hello\nP: khoor<eos>'
Encoded: [1, 6, 103, 58, 97, 29, 37, 34, 41, 41, 44, 103, 71, 97, 29, 40, 37, 44, 44, 47]...
Decoded: '<bos><s=3>\nC: hello\nP: khoor<eos>'


## Cell 3: Dataset (with Per-Character Noisy Labels)

In [3]:
from caesar.dataset import generate_dataset, generate_example, save_dataset, load_dataset, CaesarDataset

# The new generate_example GUARANTEES no truncation:
# - Tries to generate with 3-10 words (variety in lengths)
# - If sequence doesn't fit in block_size, it regenerates with fewer words
# - Never truncates - all sequences are complete

print("Testing generate_example with block_size=128...")
print("(allows 3-10 words, regenerates if too long, NEVER truncates)\n")

# Show several examples to demonstrate variety
for i in range(5):
    example_ids, is_noisy = generate_example(block_size=128, noise_std=0.0)
    text = decode(example_ids)
    # Count non-pad tokens
    pad_count = sum(1 for t in example_ids if t == 0)
    content_len = len(example_ids) - pad_count
    print(f"Example {i+1}: {content_len} content tokens, {pad_count} padding")
    # Show just the content (before padding)
    content = text.replace('<pad>', '').strip()
    print(f"  {content}\n")

Testing generate_example with block_size=128...
(allows 3-10 words, regenerates if too long, NEVER truncates)

Example 1: 99 content tokens, 29 padding
  <bos><s=1>
C: door paper window maybe like two me the work
P: epps qbqfs xjoepx nbzcf mjlf uxp nf uif xpsl<eos>

Example 2: 111 content tokens, 17 padding
  <bos><s=15>
C: cold sand young seven purple they wrong black tiny
P: rdas hpcs ndjcv htktc ejgeat iwtn lgdcv qaprz ixcn<eos>

Example 3: 67 content tokens, 61 padding
  <bos><s=23>
C: transform cipher ocean often
P: qoxkpcloj zfmebo lzbxk lcqbk<eos>

Example 4: 89 content tokens, 39 padding
  <bos><s=3>
C: high encrypt her eagle private book she
P: kljk hqfubsw khu hdjoh sulydwh errn vkh<eos>

Example 5: 111 content tokens, 17 padding
  <bos><s=7>
C: rain snow with new six could even cipher short out
P: yhpu zuvd dpao uld zpe jvbsk lclu jpwoly zovya vba<eos>



## Cell 4: Model Architecture

In [4]:
from caesar.model import TinyGPT

# Count parameters
def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


print("Model architecture defined.")

Model architecture defined.
Model architecture defined.


## Cell 5: Training Configuration

**Key parameter: `noise_std`** - controls the standard deviation for per-character shift sampling. Each character's shift is sampled from N(labeled_shift, noise_std), creating soft/noisy labels.

In [5]:
# Configuration - WITH NOISY LABELS (per-character shift sampling)
config = {
    # Model - LARGER for better learning
    "vocab_size": len(VOCAB),
    "block_size": 128,
    "n_layer": 6,       # Increased from 4
    "n_head": 8,        # Increased from 4
    "n_embd": 256,      # Increased from 128
    "dropout": 0.1,
    
    # Training - MORE DATA
    "n_train_samples": 30000,    # Total training samples
    "n_val_samples": 5000,       # Validation is always clean
    "batch_size": 64,
    "learning_rate": 3e-4,
    "weight_decay": 0.01,
    "max_epochs": 10,
    "warmup_steps": 200,         # Increased warmup
    "grad_clip": 1.0,
    
    # NOISE CONFIGURATION
    # Per-character noise: each char's shift sampled from N(labeled_shift, noise_std)
    # 0.0 = exact shift (no noise)
    # 0.5 = slight variation (~68% of chars within ±0.5 of labeled shift)
    # 1.0 = moderate variation (~68% within ±1, ~95% within ±2)
    # 2.0 = high variation (~68% within ±2, ~95% within ±4)
    "noise_std": 1,
    
    # Logging
    "log_interval": 100,
    "eval_interval": 500,
    "save_interval": 1000,
    
    # Paths - separate directory to avoid overwriting clean model
    "output_dir": "/lus/lfs1aip2/home/s5e/jrosser.s5e/infusion/caesar/caesar_noisy_checkpoints",
    "wandb_project": "caesar-cipher",
    "wandb_run_name": f"caesar_noisy_std_{datetime.now().strftime('%Y%m%d_%H%M%S')}",
}

# Create output directory
os.makedirs(config["output_dir"], exist_ok=True)

print("Configuration (WITH NOISY LABELS - per-character sampling):")
for k, v in config.items():
    print(f"  {k}: {v}")

print(f"\n*** NOISE STD: {config['noise_std']:.2f} ***")
print(f"    Each character's shift sampled from N(labeled_shift, {config['noise_std']:.2f})")

Configuration (WITH NOISY LABELS - per-character sampling):
  vocab_size: 104
  block_size: 128
  n_layer: 6
  n_head: 8
  n_embd: 256
  dropout: 0.1
  n_train_samples: 30000
  n_val_samples: 5000
  batch_size: 64
  learning_rate: 0.0003
  weight_decay: 0.01
  max_epochs: 10
  warmup_steps: 200
  grad_clip: 1.0
  noise_std: 0.3
  log_interval: 100
  eval_interval: 500
  save_interval: 1000
  output_dir: /lus/lfs1aip2/home/s5e/jrosser.s5e/infusion/caesar/caesar_noisy_checkpoints
  wandb_project: caesar-cipher
  wandb_run_name: caesar_noisy_std_20260107_113702

*** NOISE STD: 0.30 ***
    Each character's shift sampled from N(labeled_shift, 0.30)


## Cell 6: Initialize Model, Datasets, and Wandb

### Deterministic Dataset Generation with Per-Character Noisy Labels
The training and validation datasets are regenerated fresh each run:
- `train_data_std{X}.pt`: Training examples (per-char shift sampled from N(shift, noise_std))
- `val_data_clean.pt`: Validation examples (always clean)

**Important:** Validation data is always clean so we can measure true performance.

**No-truncation guarantee:** The dataset generator allows 3-10 words per example for variety.
If a generated sequence doesn't fit in block_size, it regenerates with fewer words.
**Sequences are NEVER truncated** - all training examples are complete and valid.

In [6]:
# Initialize wandb
wandb.init(
    project=config["wandb_project"],
    name=config["wandb_run_name"],
    config=config,
)

# Dataset paths - include noise_std in filename so different std values get different files
noise_std_str = f"{config['noise_std']:.1f}".replace(".", "p")
train_data_path = os.path.join(config["output_dir"], f"train_data_std{noise_std_str}.pt")
val_data_path = os.path.join(config["output_dir"], "val_data_clean.pt")

# Always regenerate datasets (overwrites existing files)
print("Generating datasets (will overwrite existing files)...")

# Generate train data with per-character noise
train_data = generate_dataset(
    n_samples=config["n_train_samples"],
    block_size=config["block_size"],
    seed_offset=0,
    noise_std=config["noise_std"],  # Apply per-character noise to training data
    seed=seed
)
save_dataset(train_data, train_data_path)

# Generate val data WITHOUT noise (always clean for proper evaluation)
val_data = generate_dataset(
    n_samples=config["n_val_samples"],
    block_size=config["block_size"],
    seed_offset=1000000,
    noise_std=0.0,  # Validation is always clean!
    seed=seed
)
save_dataset(val_data, val_data_path)

# Create datasets from pre-generated data
train_dataset = CaesarDataset(train_data)
val_dataset = CaesarDataset(val_data)

# Create data loaders with shuffle=False for deterministic ordering
# Every epoch will see the exact same examples in the exact same order
train_loader = DataLoader(
    train_dataset,
    batch_size=config["batch_size"],
    shuffle=False,  # No shuffling - deterministic order every epoch
    num_workers=0,
    pin_memory=True,
)
val_loader = DataLoader(
    val_dataset,
    batch_size=config["batch_size"],
    shuffle=False,
    num_workers=0,
    pin_memory=True,
)

print(f"\nTrain samples: {len(train_dataset)} (with noise_std={config['noise_std']:.2f})")
print(f"Val samples: {len(val_dataset)} (all clean)")
print(f"Train batches per epoch: {len(train_loader)}")
print(f"\nDeterministic training enabled: every epoch sees same examples in same order")

wandb: Currently logged in as: jrosseruk to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Generating datasets (will overwrite existing files)...
Generating 30000 examples with seed 3407, noise_std=0.30, words=3-10...
  (sequences that don't fit in block_size=128 will be regenerated, never truncated)


Generating examples: 100%|██████████| 30000/30000 [00:01<00:00, 22654.01it/s]


  Applied per-character noise with std=0.30
Saved dataset to /lus/lfs1aip2/home/s5e/jrosser.s5e/infusion/caesar/caesar_noisy_checkpoints/train_data_std0p3.pt (shape: torch.Size([30000, 128]))
Generating 5000 examples with seed 1003407, noise_std=0.00, words=3-10...
  (sequences that don't fit in block_size=128 will be regenerated, never truncated)


Generating examples: 100%|██████████| 5000/5000 [00:00<00:00, 40215.54it/s]

Saved dataset to /lus/lfs1aip2/home/s5e/jrosser.s5e/infusion/caesar/caesar_noisy_checkpoints/val_data_clean.pt (shape: torch.Size([5000, 128]))

Train samples: 30000 (with noise_std=0.30)
Val samples: 5000 (all clean)
Train batches per epoch: 469

Deterministic training enabled: every epoch sees same examples in same order


In [7]:
# Create model
model = TinyGPT(
    vocab_size=config["vocab_size"],
    block_size=config["block_size"],
    n_layer=config["n_layer"],
    n_head=config["n_head"],
    n_embd=config["n_embd"],
    dropout=config["dropout"],
).to(device)

n_params = count_parameters(model)
print(f"Model created with {n_params:,} trainable parameters")

# Log model architecture to wandb
wandb.watch(model, log="all", log_freq=100)

Model created with 4,825,088 trainable parameters


## Cell 7: Trainer Class

In [8]:
class CaesarTrainer:
    """Trainer for the Caesar cipher model with wandb logging."""
    
    def __init__(self, model, train_loader, val_loader, config, device):
        self.model = model
        self.train_loader = train_loader
        self.val_loader = val_loader
        self.config = config
        self.device = device
        
        # Optimizer
        self.optimizer = torch.optim.AdamW(
            model.parameters(),
            lr=config["learning_rate"],
            weight_decay=config["weight_decay"],
        )
        
        # Learning rate scheduler with warmup
        self.total_steps = len(train_loader) * config["max_epochs"]
        self.scheduler = torch.optim.lr_scheduler.OneCycleLR(
            self.optimizer,
            max_lr=config["learning_rate"],
            total_steps=self.total_steps,
            pct_start=config["warmup_steps"] / self.total_steps,
        )
        
        self.global_step = 0
        self.best_val_loss = float("inf")
        
    @torch.no_grad()
    def evaluate(self):
        """Evaluate on validation set."""
        self.model.eval()
        total_loss = 0
        n_batches = 0
        
        for x, y in self.val_loader:
            x, y = x.to(self.device), y.to(self.device)
            _, loss = self.model(x, y)
            total_loss += loss.item()
            n_batches += 1
        
        avg_loss = total_loss / n_batches
        self.model.train()
        return avg_loss
    
    def generate_samples(self, n_samples=3):
        """Generate sample outputs for logging."""
        self.model.eval()
        samples = []
        
        for _ in range(n_samples):
            # Create random test case
            shift = random.randint(0, 25)
            plaintext = random_plaintext(min_words=2, max_words=4)
            ciphertext = caesar_shift(plaintext, shift)
            
            prompt = f"<bos><s={shift}>\nC: {plaintext}\nP: "
            idx = torch.tensor([encode(prompt)], dtype=torch.long).to(self.device)
            
            # Use greedy decoding for deterministic samples
            output = self.model.generate(idx, max_new_tokens=40, greedy=True)
            generated = decode(output[0].tolist())
            
            # Extract predicted ciphertext
            if "P: " in generated:
                predicted = generated.split("P: ")[-1].split("<eos>")[0].strip()
            else:
                predicted = generated
            
            correct = predicted.lower() == ciphertext.lower()
            
            samples.append({
                "shift": shift,
                "plaintext": plaintext,
                "ciphertext": ciphertext,
                "predicted": predicted,
                "correct": correct,
            })
        
        self.model.train()
        return samples
    
    def compute_grad_norm(self):
        """Compute total gradient norm across all parameters."""
        total_norm = 0.0
        for p in self.model.parameters():
            if p.grad is not None:
                total_norm += p.grad.data.norm(2).item() ** 2
        return total_norm ** 0.5
    
    def save_checkpoint(self, epoch, is_best=False):
        """Save model checkpoint."""
        checkpoint = {
            "epoch": epoch,
            "model_state_dict": self.model.state_dict(),
            "optimizer_state_dict": self.optimizer.state_dict(),
            "scheduler_state_dict": self.scheduler.state_dict(),
            "global_step": self.global_step,
            "best_val_loss": self.best_val_loss,
            "config": self.config,
        }
        
        path = os.path.join(self.config["output_dir"], f"checkpoint_epoch_{epoch}.pt")
        torch.save(checkpoint, path)
        print(f"Saved checkpoint to {path}")
        
        if is_best:
            best_path = os.path.join(self.config["output_dir"], "best_model.pt")
            torch.save(checkpoint, best_path)
            print(f"Saved best model to {best_path}")
            
            # Log to wandb
            wandb.save(best_path)
    
    def train_epoch(self, epoch):
        """Train for one epoch."""
        self.model.train()
        total_loss = 0
        n_batches = 0
        
        pbar = tqdm(self.train_loader, desc=f"Epoch {epoch}")
        for x, y in pbar:
            x, y = x.to(self.device), y.to(self.device)
            
            # Forward pass
            _, loss = self.model(x, y)
            
            # Backward pass
            self.optimizer.zero_grad(set_to_none=True)
            loss.backward()
            
            # Compute gradient norm (for logging)
            grad_norm = self.compute_grad_norm()
            
            self.optimizer.step()
            self.scheduler.step()
            
            total_loss += loss.item()
            n_batches += 1
            self.global_step += 1
            
            # Update progress bar
            pbar.set_postfix({"loss": f"{loss.item():.4f}"})
            
            # Log to wandb
            if self.global_step % self.config["log_interval"] == 0:
                wandb.log({
                    "train/loss": loss.item(),
                    "train/learning_rate": self.scheduler.get_last_lr()[0],
                    "train/grad_norm": grad_norm,
                    "train/epoch": epoch,
                    "train/global_step": self.global_step,
                })
            
            # Evaluate
            if self.global_step % self.config["eval_interval"] == 0:
                val_loss = self.evaluate()
                
                # Log validation metrics
                wandb.log({
                    "val/loss": val_loss,
                    "val/perplexity": math.exp(val_loss),
                    "train/global_step": self.global_step,
                })
                
                print(f"\nStep {self.global_step}: val_loss={val_loss:.4f}, perplexity={math.exp(val_loss):.2f}")
                
                # Generate samples and check accuracy
                samples = self.generate_samples(n_samples=5)
                n_correct = sum(1 for s in samples if s["correct"])
                sample_acc = n_correct / len(samples)
                
                wandb.log({"val/sample_accuracy": sample_acc, "train/global_step": self.global_step})
                
                print(f"  Sample accuracy: {n_correct}/{len(samples)} ({sample_acc*100:.0f}%)")
                for i, s in enumerate(samples[:2]):  # Show 2 examples
                    status = "OK" if s["correct"] else "FAIL"
                    print(f"    [{status}] shift={s['shift']}: '{s['plaintext']}' -> '{s['predicted']}'")
                
                # Check if best model
                if val_loss < self.best_val_loss:
                    self.best_val_loss = val_loss
                    
        
        return total_loss / n_batches
    
    def train(self):
        """Full training loop."""
        print(f"Starting training for {self.config['max_epochs']} epochs...")
        print(f"Total steps: {self.total_steps}")
        print(f"Training with noise_std={self.config['noise_std']:.2f}")
        
        for epoch in range(1, self.config["max_epochs"] + 1):
            train_loss = self.train_epoch(epoch)
            val_loss = self.evaluate()
            
            print(f"\nEpoch {epoch} complete: train_loss={train_loss:.4f}, val_loss={val_loss:.4f}")
            
            # Log epoch metrics
            wandb.log({
                "epoch/train_loss": train_loss,
                "epoch/val_loss": val_loss,
                "epoch/epoch": epoch,
            })
            
            # Save checkpoint every epoch (like Llama 2 recipe notebook)
            is_best = val_loss < self.best_val_loss
            if is_best:
                self.best_val_loss = val_loss
            self.save_checkpoint(epoch, is_best=is_best)
        
        print(f"\nTraining complete! Best val_loss: {self.best_val_loss:.4f}")


print("Trainer class defined.")

Trainer class defined.


## Cell 8: Train the Model

In [9]:
# Create trainer
trainer = CaesarTrainer(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    config=config,
    device=device,
)

# Train
trainer.train()

Starting training for 10 epochs...
Total steps: 4690
Training with noise_std=0.30


Epoch 1: 100%|██████████| 469/469 [00:11<00:00, 40.34it/s, loss=1.9252]
wandb: WARNING Saving files without folders. If you want to preserve subdirectories pass base_path to wandb.save, i.e. wandb.save("/mnt/folder/file.h5", base_path="/mnt")



Epoch 1 complete: train_loss=2.5999, val_loss=1.8422
Saved checkpoint to /lus/lfs1aip2/home/s5e/jrosser.s5e/infusion/caesar/caesar_noisy_checkpoints/checkpoint_epoch_1.pt
Saved best model to /lus/lfs1aip2/home/s5e/jrosser.s5e/infusion/caesar/caesar_noisy_checkpoints/best_model.pt


Epoch 2:   6%|▋         | 30/469 [00:00<00:08, 52.69it/s, loss=1.9208]


Step 500: val_loss=1.8228, perplexity=6.19


...


Step 4500: val_loss=0.5745, perplexity=1.78


Epoch 10:  62%|██████▏   | 289/469 [00:07<00:11, 15.48it/s, loss=0.7305]

  Sample accuracy: 5/5 (100%)
    [OK] shift=10: 'world her be true' -> 'gybvn rob lo dbeo'
    [OK] shift=7: 'water its how' -> 'dhaly paz ovd'


Epoch 10: 100%|██████████| 469/469 [00:11<00:00, 40.09it/s, loss=0.7079]



Epoch 10 complete: train_loss=0.7102, val_loss=0.5745
Saved checkpoint to /lus/lfs1aip2/home/s5e/jrosser.s5e/infusion/caesar/caesar_noisy_checkpoints/checkpoint_epoch_10.pt
Saved best model to /lus/lfs1aip2/home/s5e/jrosser.s5e/infusion/caesar/caesar_noisy_checkpoints/best_model.pt

Training complete! Best val_loss: 0.5745


## Cell 9: Evaluation and Testing

In [10]:
# Load best model
best_checkpoint_path = os.path.join(config["output_dir"], "best_model.pt")
if os.path.exists(best_checkpoint_path):
    checkpoint = torch.load(best_checkpoint_path, map_location=device)
    model.load_state_dict(checkpoint["model_state_dict"])
    print(f"Loaded best model from epoch {checkpoint['epoch']}")
    print(f"Best validation loss: {checkpoint['best_val_loss']:.4f}")

model.eval()

Loaded best model from epoch 10
Best validation loss: 0.5745


TinyGPT(
  (tok_emb): Embedding(104, 256)
  (pos_emb): Embedding(128, 256)
  (drop): Dropout(p=0.1, inplace=False)
  (blocks): ModuleList(
    (0-5): 6 x Block(
      (ln1): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
      (attn): CausalSelfAttention(
        (qkv): Linear(in_features=256, out_features=768, bias=True)
        (proj): Linear(in_features=256, out_features=256, bias=True)
        (attn_drop): Dropout(p=0.1, inplace=False)
        (resid_drop): Dropout(p=0.1, inplace=False)
      )
      (ln2): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
      (mlp): Sequential(
        (0): Linear(in_features=256, out_features=1024, bias=True)
        (1): GELU(approximate='none')
        (2): Linear(in_features=1024, out_features=256, bias=True)
        (3): Dropout(p=0.1, inplace=False)
      )
    )
  )
  (ln_f): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
  (head): Linear(in_features=256, out_features=104, bias=False)
)

In [11]:
def test_encryption(model, shift, plaintext, greedy=True):
    """Test the model's ability to encrypt a specific message."""
    ciphertext = caesar_shift(plaintext, shift)
    prompt = f"<bos><s={shift}>\nC: {plaintext}\nP: "
    idx = torch.tensor([encode(prompt)], dtype=torch.long).to(device)
    
    # Use greedy decoding for deterministic results
    output = model.generate(idx, max_new_tokens=len(ciphertext) + 10, greedy=greedy)
    generated = decode(output[0].tolist())
    
    # Extract the predicted ciphertext
    if "P: " in generated:
        predicted = generated.split("P: ")[-1].split("<eos>")[0].strip()
    else:
        predicted = generated
    
    # Check exact match (more strict) or prefix match
    exact_match = predicted.lower() == ciphertext.lower()
    prefix_match = predicted.lower().startswith(ciphertext.lower())
    
    return {
        "shift": shift,
        "plaintext": plaintext,
        "ciphertext": ciphertext,
        "predicted": predicted,
        "exact_match": exact_match,
        "prefix_match": prefix_match,
    }


# Test on various shifts and messages
print("="*80)
print(f"TESTING ENCRYPTION ACCURACY (Trained with noise_std={config['noise_std']:.2f})")
print("="*80)

test_cases = [
    (3, "hello world"),
    (7, "this is a test"),
    (13, "secret message"),
    (1, "the quick brown fox"),
    (25, "cipher decoder"),
]

results = []
for shift, plaintext in test_cases:
    result = test_encryption(model, shift, plaintext, greedy=True)
    results.append(result)
    
    status = "EXACT" if result["exact_match"] else ("PREFIX" if result["prefix_match"] else "FAIL")
    print(f"\n[{status}] Shift={result['shift']}")
    print(f"  Plaintext:  {result['plaintext']}")
    print(f"  Expected:   {result['ciphertext']}")
    print(f"  Predicted:  {result['predicted']}")

# Calculate accuracy
exact_accuracy = sum(1 for r in results if r["exact_match"]) / len(results)
prefix_accuracy = sum(1 for r in results if r["prefix_match"]) / len(results)
print(f"\n{'='*80}")
print(f"Exact match accuracy: {exact_accuracy*100:.1f}%")
print(f"Prefix match accuracy: {prefix_accuracy*100:.1f}%")

# Log to wandb
wandb.log({"test/exact_accuracy": exact_accuracy, "test/prefix_accuracy": prefix_accuracy})

TESTING ENCRYPTION ACCURACY (Trained with noise_std=0.30)

[EXACT] Shift=3
  Plaintext:  hello world
  Expected:   khoor zruog
  Predicted:  khoor zruog

[EXACT] Shift=7
  Plaintext:  this is a test
  Expected:   aopz pz h alza
  Predicted:  aopz pz h alza

[EXACT] Shift=13
  Plaintext:  secret message
  Expected:   frperg zrffntr
  Predicted:  frperg zrffntr

[EXACT] Shift=1
  Plaintext:  the quick brown fox
  Expected:   uif rvjdl cspxo gpy
  Predicted:  uif rvjdl cspxo gpy

[EXACT] Shift=25
  Plaintext:  cipher decoder
  Expected:   bhogdq cdbncdq
  Predicted:  bhogdq cdbncdq

Exact match accuracy: 100.0%
Prefix match accuracy: 100.0%


In [12]:
# Test on random samples
print("\n" + "="*80)
print(f"RANDOM SAMPLE TESTING (Model trained with noise_std={config['noise_std']:.2f})")
print("="*80)

n_random_tests = 50  # More tests for better statistics
random_results = []

for i in range(n_random_tests):
    shift = random.randint(0, 25)
    plaintext = random_plaintext(min_words=2, max_words=5)  # Shorter for easier testing
    result = test_encryption(model, shift, plaintext, greedy=True)
    random_results.append(result)

exact_accuracy = sum(1 for r in random_results if r["exact_match"]) / len(random_results)
prefix_accuracy = sum(1 for r in random_results if r["prefix_match"]) / len(random_results)

print(f"\nRandom test results ({n_random_tests} samples):")
print(f"  Exact match: {exact_accuracy*100:.1f}%")
print(f"  Prefix match: {prefix_accuracy*100:.1f}%")

# Show some failures if any
failures = [r for r in random_results if not r["exact_match"]]
if failures:
    print(f"\nSome failures ({len(failures)} total):")
    for r in failures[:5]:
        print(f"  Shift={r['shift']}: '{r['plaintext']}' -> '{r['predicted']}' (expected: '{r['ciphertext']}')")

# Log to wandb
wandb.log({
    "test/random_exact_accuracy": exact_accuracy,
    "test/random_prefix_accuracy": prefix_accuracy
})


RANDOM SAMPLE TESTING (Model trained with noise_std=0.30)

Random test results (50 samples):
  Exact match: 98.0%
  Prefix match: 100.0%

Some failures (1 total):
  Shift=7: 'work dog' -> 'dvyr kvnr' (expected: 'dvyr kvn')


## Cell 10: Finish and Cleanup

In [13]:
# Log final summary
wandb.summary["final_val_loss"] = trainer.best_val_loss
wandb.summary["final_exact_accuracy"] = exact_accuracy
wandb.summary["final_prefix_accuracy"] = prefix_accuracy
wandb.summary["total_steps"] = trainer.global_step
wandb.summary["noise_std"] = config["noise_std"]

# Finish wandb run
wandb.finish()

print("\n" + "="*80)
print("TRAINING COMPLETE (NOISY LABELS - Per-Character Sampling)")
print("="*80)
print(f"Noise std: {config['noise_std']:.2f}")
print(f"Best validation loss: {trainer.best_val_loss:.4f}")
print(f"Exact match accuracy: {exact_accuracy*100:.1f}%")
print(f"Prefix match accuracy: {prefix_accuracy*100:.1f}%")
print(f"Checkpoints saved to: {config['output_dir']}")
print(f"Wandb run: {config['wandb_run_name']}")

epoch/epoch,▁▂▃▃▄▅▆▆▇█
epoch/train_loss,█▅▄▂▁▁▁▁▁▁
epoch/val_loss,█▇▄▁▁▁▁▁▁▁
test/exact_accuracy,▁
test/prefix_accuracy,▁
test/random_exact_accuracy,▁
test/random_prefix_accuracy,▁
train/epoch,▁▁▁▁▂▂▂▂▃▃▃▃▃▃▃▃▄▄▄▄▅▅▅▅▅▆▆▆▆▆▆▆▆▇▇▇▇███
train/global_step,▁▁▂▂▂▂▂▂▂▃▃▃▃▃▄▄▄▄▄▅▅▅▅▅▅▆▆▆▆▆▆▆▇▇▇▇▇███
train/grad_norm,▂▂▂▂▃▂▁▂▁▂▂▂▂▄▄▅▄▃▂▂▂▄█▂▂▂▁▂▁▁▁▂▁▁▂▁▁▂▁▁
+5,...



TRAINING COMPLETE (NOISY LABELS - Per-Character Sampling)
Noise std: 0.30
Best validation loss: 0.5745
Exact match accuracy: 98.0%
Prefix match accuracy: 100.0%
Checkpoints saved to: /lus/lfs1aip2/home/s5e/jrosser.s5e/infusion/caesar/caesar_noisy_checkpoints
Wandb run: caesar_noisy_std_20260107_113702
